### Importing Files

In [20]:
import torch 
import torch_geometric

from torch_geometric.nn import MessagePassing
import torch.nn.functional as F
from torch_geometric.utils import dense_to_sparse, softmax

### Common declarations

In [21]:
labels = torch.tensor([0, 0, 0, 1, 1])
criterion = torch.nn.CrossEntropyLoss()

In [22]:
X_sample = torch.randn(5,6)
# print(X)


coords = torch.tensor([
    [0.0,  0.7,  0.0],   # Fz
    [0.3,  0.5,  0.0],   # F3
   [-0.3,  0.5,  0.0],   # F4
    [0.0,  0.0,  0.9],   # Cz
    [0.5,  0.0,  0.5],   # C4
])

node_names = ["Fz", "F3", "F4", "CZ", "C4"]

W = torch.randn(6,4)

A = torch.zeros(5,5)

def check_neighbors():
    for electrode in range(len(coords)): # Running over each electrode's coordinates
        for neighbor in range(len(coords)):
            if neighbor != electrode:
                distance_from_neighbor = 0.0
                for coord in range(len(coords[electrode])): # Running over each coordinate of electrode and the comparison electrode
                    distance_from_neighbor += (coords[electrode][coord] - coords[neighbor][coord]) ** 2
                
                distance_from_neighbor = distance_from_neighbor ** 0.5

            else:
                continue

            # print(f"Distance between: {node_names[electrode]} and {node_names[neighbor]} is: {distance_from_neighbor}")

            if distance_from_neighbor < 0.61:
                print(f"{node_names[electrode]} and {node_names[neighbor]} are neighbours")
            
            # else:
            #     print(f"{node_names[electrode]} and {node_names[neighbor]} are not neighbours")
            #     print(distance_from_neighbor)

In [23]:
class GCNLayer(torch.nn.Module):
    def __init__(self, in_features, out_features):
        super().__init__()
        self.W = torch.nn.Parameter(torch.randn(in_features, out_features))
        self.in_features = in_features
        self.out_features = out_features
    
    def forward(self, X, A):
        degrees = A.sum(dim=1)
        D_inv = torch.diag(1.0/degrees)

        normalized_A = torch.matmul(D_inv, A)
        normalized_result = torch.matmul(normalized_A, X)

        transformed_results = torch.matmul(normalized_result, self.W)

        return transformed_results

In [24]:
#Creating adjacency matrix
diff = coords.unsqueeze(0) - coords.unsqueeze(1) # Converts tensors to single dimensional tensors
dist = (diff ** 2).sum(dim=-1) ** 0.5  
A = (dist < 0.61).float() # Condition for each value in the tensor. if true => 1., if not => 0.
A.fill_diagonal_(0) # Fills diagonals with zeros

# Fixing degree-scale problem
identity_matrix = torch.eye(5)
A_hat = A + identity_matrix

edge_index, _ = dense_to_sparse(A_hat)

In [25]:
gcn = GCNLayer(6, 4)
optimizer_gcn = torch.optim.Adam(gcn.parameters(), lr = 0.01)

# for epoch in range(100):
#     optimizer.zero_grad()
#     out = gcn(X_sample, A_hat)
#     loss = criterion(out, labels)
#     loss.backward()
#     optimizer.step()

    # if epoch % 10 == 0:
    #     print(f"Epoch {epoch}, Loss: {loss.item()}")

In [26]:
class GATLayer(MessagePassing):
    def __init__(self, in_features, out_features):
        super().__init__(aggr='add')
        self.W = torch.nn.Linear(in_features, out_features, bias = False) #Wraps X @ W into cleaner syntax
        self.a = torch.nn.Parameter(torch.randn(2 * out_features)) # fEATURE vector for the two nodes

    def forward(self, X, edge_index):
        X = self.W(X)
        return self.propagate(edge_index, X=X, size=(X.shape[0], X.shape[0]))
    
    def message(self, X_i, X_j, index):
        cont_msg = torch.cat([X_i, X_j], dim=1)
        att_score = (self.a * cont_msg).sum(dim=-1, keepdim=True)
        final_score = softmax(att_score, index)

        return X_j * final_score

    def update(self, aggr_out):
        return aggr_out

In [27]:
# gat= GATLayer(6,4)
# optimizer = torch.optim.Adam(gat.parameters(), lr = 0.01)

# for epoch in range(1000):
#     optimizer.zero_grad()
#     out = gat(X_sample, edge_index)
#     loss = criterion(out, labels)
#     loss.backward()
#     optimizer.step()

#     if epoch % 200 == 0:
#         print(f"Epoch {epoch}, Loss: {loss.item()}")

In [28]:
class BrainSync(torch.nn.Module):
    def __init__(self, in_features, out_features):
        super().__init__()
        self.gat = GATLayer(in_features, out_features)
        self.lin_layer = torch.nn.Linear(out_features, 2)

    def forward(self, X, edge_index):
        out = self.gat(X, edge_index)
        mean = torch.mean(out, dim=0)
        classes = self.lin_layer(mean)

        return classes

In [29]:
# X = torch.randn(64, 6)
# edge_index_final = torch.randint(0, 64, (2, 200))
# labels_final = torch.randint(0, 2, (1,)).squeeze() #for the graph not the features

# sync = BrainSync(6, 4)

# optimizer = torch.optim.Adam(sync.parameters(), lr = 0.01)

# for epoch in range(100):
#     optimizer.zero_grad()
#     out = sync(X, edge_index_final)
#     loss = criterion(out, labels_final)
#     loss.backward()
#     optimizer.step()

#     # if epoch % 10 == 0:
#     #     print(f"Epoch {epoch}, Loss: {loss.item()}")

In [30]:
class BrainSync_2(torch.nn.Module):
    def __init__(self, in_features, out_features):
        super().__init__()
        self.gat_layer1 = GATLayer(in_features, 4)
        self.gat_layer2 = GATLayer(4, out_features)
        self.lin_layer = torch.nn.Linear(out_features, 2)

    def forward(self, X, edge_index):
        res1 = self.gat_layer1(X, edge_index)
        res2 = self.gat_layer2(res1, edge_index)
        mean = torch.mean(res2, dim=0)
        classes = self.lin_layer(mean)
        
        return classes

In [31]:
# sync_2layered = BrainSync_2(6,4)
# optimizer = torch.optim.Adam(sync_2layered.parameters(), lr = 0.01)
# X = torch.randn(64, 6)
# edge_index_final = torch.randint(0, 64, (2, 200))
# labels_final = torch.randint(0, 2, (1,)).squeeze() #for the graph not the features

# for epoch in range(100):
#     optimizer.zero_grad()
#     out = sync_2layered(X, edge_index_final)
#     loss = criterion(out, labels_final)
#     loss.backward()
#     optimizer.step()

#     # if epoch % 10 == 0:
#     #     print(f"Epoch {epoch}, Loss: {loss.item()}")

In [32]:
class BrainSync_8(torch.nn.Module):
    def __init__(self, in_features, out_features):
        super().__init__()
        
        self.layer1 = GATLayer(in_features, 4)
        self.layer2 = GATLayer(4, 4)
        self.layer3 = GATLayer(4, 4)
        self.layer4 = GATLayer(4, 4)
        self.layer5 = GATLayer(4, 4)
        self.layer6 = GATLayer(4, 4)
        self.layer7 = GATLayer(4, 4)
        self.layer8 = GATLayer(4, out_features)

        self.proj = torch.nn.Linear(in_features, 4)
        self.lin_layer = torch.nn.Linear(out_features, 2)


    def forward(self, X, edge_index):
        res1 = self.layer1(X, edge_index)
        prev_proj = self.proj(X)
        res1 = res1 + prev_proj
        res2 = self.layer2(res1, edge_index)
        res2 = res2 + res1
        res3 = self.layer3(res2, edge_index)
        res3 = res3 + res2
        res4 = self.layer4(res3, edge_index)
        res4 = res4 + res3
        res5 = self.layer5(res4, edge_index)
        res5 = res5 + res4
        res6 = self.layer6(res5, edge_index)
        res6 = res6 + res5
        res7 = self.layer7(res6, edge_index)
        res7 = res7 + res6
        res8 = self.layer8(res7, edge_index)
        res8 = res8 + res7

        mean = torch.mean(res8, dim=0)

        classes = self.lin_layer(mean)

        return classes

In [33]:
# gat_8layered = BrainSync_8(6,4)
# optimizer = torch.optim.Adam(gat_8layered.parameters(), lr=0.0001)

# X = torch.randn(64, 6)
# edge_index_final = torch.randint(0, 64, (2, 200))
# labels_final = torch.randint(0, 2, (1,)).squeeze() #for the graph not the features

# for epoch in range(100):
#     optimizer.zero_grad()
#     out = gat_8layered(X, edge_index_final)
#     print(out)
#     loss = criterion(out, labels_final)
#     print(labels_final)
#     loss.backward()
#     optimizer.step()

#     if epoch % 10 == 0:
#         print(f"Epoch {epoch}, Loss: {loss.item()}")

In [34]:
# with torch.no_grad():
#     X = torch.randn(64, 6)
#     h = X.clone()
#     for layer in [gat_8layered.layer1, gat_8layered.layer2, gat_8layered.layer3,
#                   gat_8layered.layer4, gat_8layered.layer5, gat_8layered.layer6,
#                   gat_8layered.layer7, gat_8layered.layer8]:
#         h = layer(h, edge_index_final)
#         print(f"Std across nodes: {h.std(dim=0).mean().item():.6f}")

In [35]:
test_layer = GATLayer(6, 4)
X_test = torch.randn(10, 6)
edge_test = torch.randint(0, 10, (2, 20))
out = test_layer(X_test, edge_test)
print(out)

tensor([[-0.1417,  0.5760, -0.1672, -0.3018],
        [ 0.2545,  0.0377,  0.0129, -0.0643],
        [ 0.0000,  0.0000,  0.0000,  0.0000],
        [-0.3027,  0.5133, -0.1727, -0.1970],
        [-0.2235,  0.2694, -0.1252, -0.0670],
        [ 0.0000,  0.0000,  0.0000,  0.0000],
        [-0.1924,  0.1215, -0.1334,  0.0841],
        [-0.0167,  0.4044,  0.0732, -0.0499],
        [-0.1176, -0.2652,  0.0222,  0.1636],
        [-0.0349,  0.3752,  0.3922,  0.2274]], grad_fn=<ScatterAddBackward0>)


In [36]:
gat_8layered_skip = BrainSync_8(6, 4)
optimizer = torch.optim.Adam(gat_8layered_skip.parameters(), lr=0.01)

In [37]:
X = torch.randn(64, 6)

edge_index_final_skip = torch.randint(0, 64, (2, 200))
labels_final_skip = torch.randint(0, 2, (1,)).squeeze() #for the graph not the features

for epoch in range(100):
    optimizer.zero_grad()
    out = gat_8layered_skip(X, edge_index_final_skip)
    loss = criterion(out, labels_final_skip)
    # print(f"out: {out} | loss: {loss}")
    loss.backward()
    optimizer.step()

    if epoch % 10 == 0: 
        print(f"Epoch {epoch}, Loss: {loss.item()}")



Epoch 0, Loss: 0.468313992023468
Epoch 10, Loss: 0.00018654513405635953
Epoch 20, Loss: 0.0
Epoch 30, Loss: 0.0
Epoch 40, Loss: 0.0
Epoch 50, Loss: 0.0
Epoch 60, Loss: 0.0
Epoch 70, Loss: 0.0
Epoch 80, Loss: 0.0
Epoch 90, Loss: 0.0


In [38]:
with torch.no_grad():
    X = torch.randn(64, 6)
    h = X.clone()
    proj_X = gat_8layered_skip.proj(X)
    
    h = gat_8layered_skip.layer1(h, edge_index_final_skip) + proj_X
    print(f"Layer 1: {h.std(dim=0).mean().item():.6f}")
    
    for i, layer in enumerate([gat_8layered_skip.layer2, gat_8layered_skip.layer3,
                                gat_8layered_skip.layer4, gat_8layered_skip.layer5,
                                gat_8layered_skip.layer6, gat_8layered_skip.layer7,
                                gat_8layered_skip.layer8]):
        prev = h.clone()
        h = layer(h, edge_index_final_skip) + prev
        print(f"Layer {i+2}: {h.std(dim=0).mean().item():.6f}")

Layer 1: 0.761776
Layer 2: 0.949388
Layer 3: 1.233937
Layer 4: 1.614294
Layer 5: 1.707807
Layer 6: 2.544670
Layer 7: 3.131418
Layer 8: 3.649880
